In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import torch                       # <--- MỚI: Thêm PyTorch
import torch.nn as nn              # <--- MỚI
import torch.optim as optim        # <--- MỚI
import torchvision                 # <--- MỚI
import torchvision.transforms as transforms # <--- MỚI
from scipy.signal import butter, filtfilt, hilbert
import os
import pickle
import gc
import json

CFG = {
    "fs": 1.0,            # <--- SỬA: Coi mỗi ảnh là 1 bước thời gian (sample rate = 1)
    "batch_size": 256,    # <--- MỚI: Đóng vai trò là "Time duration" cho thuật toán Psi
    "num_classes": 10,    # <--- MỚI: CIFAR-10 có 10 lớp
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "legend.frameon": False,
    }
)

def format_label(key: str) -> str:
    """
    Convert internal identifiers to formatted labels.
    """
    special_map = {
        "pac": "PAC"
    }

    words = key.replace("_", " ").split()
    pretty_words = []
    for w in words:
        lw = w.lower()
        if lw in special_map:
            pretty_words.append(special_map[lw])
        elif w.isupper():
            pretty_words.append(w)
        else:
            pretty_words.append(w.capitalize())
    return " ".join(pretty_words)

In [2]:
def setup_resnet_hooks(model):
    """
    Gắn hook vào 4 tầng chính của ResNet18 để lấy dữ liệu đa cấp độ.
    Tương ứng với các blocks {1, 4, 7, 10} trong bài báo GPT-2.
    """
    activations = {}
    
    def get_activation(name):
        def hook(model, input, output):
            # Chỉ giữ lại tensor detached để tiết kiệm VRAM
            activations[name] = output.detach()
        return hook

    # ResNet18 có 4 lớp conv chính: layer1, layer2, layer3, layer4
    layers_to_hook = ['layer1', 'layer2', 'layer3', 'layer4']
    
    for name in layers_to_hook:
        # Kiểm tra xem model có layer đó không (đề phòng sửa kiến trúc)
        if hasattr(model, name):
            getattr(model, name).register_forward_hook(get_activation(name))
            
    return activations

def gather_multiscale_activations(activations_dict, samples_per_layer=32):
    """
    Thực hiện ALGORITHM 1 (Lines 7-9):
    1. Lấy activation từ 4 layers.
    2. Global Average Pooling (giảm chiều không gian H,W).
    3. Random sample 32 channels/layer.
    4. Concatenate thành matrix [Batch, 128].
    """
    collected_features = []
    layers = ['layer1', 'layer2', 'layer3', 'layer4']
    
    for layer_name in layers:
        if layer_name not in activations_dict:
            continue
            
        # Raw shape: [Batch, Channels, H, W]
        act = activations_dict[layer_name]
        
        # Global Average Pooling: [B, C, H, W] -> [B, C]
        # Đại diện cho trạng thái kích hoạt trung bình của feature map
        if len(act.shape) > 2:
            act = torch.mean(act, dim=[2, 3]) 
            
        # Sampling: Lấy ngẫu nhiên 32 channels
        n_channels = act.shape[1]
        
        # Dùng generator để đảm bảo tính ngẫu nhiên nhưng đồng nhất trong 1 batch
        if n_channels >= samples_per_layer:
            idx = torch.randperm(n_channels, device=act.device)[:samples_per_layer]
            act = act[:, idx]
        else:
            # Trường hợp hiếm (layer quá nhỏ): giữ nguyên
            pass
            
        collected_features.append(act)
    
    # Concatenate theo chiều Channels (dim 1) -> [Batch, 128]
    if not collected_features:
        return None
        
    final_matrix = torch.cat(collected_features, dim=1)
    return final_matrix

def prepare_cnn_activations(activations, labels, smooth_window=5):
    """
    ADAPTER V3 (FINAL):
    Biến đổi Matrix Activation thành dạng sóng giả lập (Pseudo-time Series).
    Bao gồm: Sắp xếp theo Class -> Local Centering -> Smoothing.
    
    Input: activations [Batch, 128], labels [Batch]
    Output: numpy array [128, Batch] (Channels x Time)
    """
    # 1. Chuyển về CPU numpy
    if isinstance(activations, torch.Tensor):
        data = activations.detach().cpu().numpy()
    else:
        data = activations
        
    if isinstance(labels, torch.Tensor):
        lbls = labels.detach().cpu().numpy()
    else:
        lbls = labels
    
    # 2. Sắp xếp theo nhãn (Sort by Label) -> Tạo cấu trúc thời gian giả lập
    sort_idx = np.argsort(lbls)
    data_sorted = data[sort_idx] # Shape: (Time, Channels)
    
    # 3. Class-wise Centering (Quan trọng cho Metastability)
    # Loại bỏ giá trị trung bình của từng class để làm nổi bật dao động
    df_temp = pd.DataFrame(data_sorted)
    df_temp['label'] = lbls[sort_idx]
    
    # Trừ đi mean của từng nhóm class
    data_detrended = df_temp.groupby('label').transform(lambda x: x - x.mean()).values
    
    # 4. Smoothing (Quan trọng cho Hurst)
    # Khử nhiễu gai góc của ReLU, giúp H tăng lên mức 0.6-0.8
    # Window=5 là đủ để mượt mà vẫn giữ được biến động nhanh
    data_smoothed = pd.DataFrame(data_detrended).rolling(
        window=smooth_window, center=True, min_periods=1
    ).mean().values
    
    # 5. Transpose về (Channels, Time) để phù hợp hàm DFA/Hilbert
    return data_smoothed.T

In [3]:
# ---------------------------------------------------------------------
# CORE METRIC UTILITIES (DFA, LZ, MI)
# ---------------------------------------------------------------------

def dfa_hurst(x, min_win=16, max_win=None, n_win=10):
    """
    DFA-based Hurst exponent.
    SỬA ĐỔI: Thêm check độ dài dữ liệu để tránh crash nếu Batch Size nhỏ.
    """
    x = np.asarray(x)
    N = x.size
    
    # Safety check: Nếu Batch Size < 32, DFA không tính được chính xác -> Trả về 0.5 (Random walk)
    if N < 32: 
        return 0.5
        
    if max_win is None:
        max_win = N // 4
        
    # Safety check: Đảm bảo max_win luôn lớn hơn min_win
    if max_win <= min_win:
        max_win = N // 2

    y = np.cumsum(x - x.mean())
    
    # Tạo danh sách các cửa sổ (log-scale)
    s_vals = np.unique(
        np.logspace(np.log10(min_win), np.log10(max_win), n_win, dtype=int)
    )
    
    F = []
    for s in s_vals:
        if s < 4: continue
        
        n_segments = N // s
        if n_segments < 2: continue
        
        rms = []
        for i in range(n_segments):
            seg = y[i * s : (i + 1) * s]
            t = np.arange(s)
            p = np.polyfit(t, seg, 1)
            trend = np.polyval(p, t)
            detrended = seg - trend
            rms.append(np.sqrt(np.mean(detrended**2)))
            
        if rms:
            F.append(np.mean(rms))
            
    F = np.array(F)
    if len(F) < 2:
        return 0.5
    F = np.where(F <= 0, 1e-10, F)
    s_use = s_vals[: len(F)]
    # Fit đường thẳng trên đồ thị log-log
    coeffs = np.polyfit(np.log(s_use), np.log(F), 1)
    H = coeffs[0]
    return float(H)


def lempel_ziv_complexity(binary_seq):
    """
    Lempel–Ziv complexity for a 1D binary sequence (0/1).
    Giữ nguyên.
    """
    s = "".join(str(int(b)) for b in binary_seq)
    i, c, l = 0, 1, 1
    n = len(s)
    while True:
        if i + l > n:
            c += 1
            break
        sub = s[i : i + l]
        if sub in s[:i]:
            l += 1
        else:
            i += l
            c += 1
            l = 1
        if i + l > n:
            break
    return c / n


def mutual_information_phase_amp(phase, amp, n_bins=12):
    """
    Mutual information between phase and amplitude.
    Giữ nguyên (có thể dùng để tính toán phụ, dù Psi chính chỉ cần H và M).
    """
    phase = np.asarray(phase)
    amp = np.asarray(amp)

    phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    amp_bins = np.quantile(amp, np.linspace(0, 1, n_bins + 1))
    phase_d = np.digitize(phase, phase_bins) - 1
    amp_d = np.digitize(amp, amp_bins) - 1

    valid = (
        (phase_d >= 0) & (phase_d < n_bins) & 
        (amp_d >= 0) & (amp_d < n_bins)
    )
    phase_d = phase_d[valid]
    amp_d = amp_d[valid]
    
    if len(phase_d) == 0: return 0.0

    joint, _, _ = np.histogram2d(phase_d, amp_d, bins=(n_bins, n_bins))
    if joint.sum() == 0: return 0.0
    
    joint = joint / joint.sum()
    px = joint.sum(axis=1)
    py = joint.sum(axis=0)

    mi = 0.0
    for i in range(n_bins):
        for j in range(n_bins):
            pxy = joint[i, j]
            if pxy > 0 and px[i] > 0 and py[j] > 0:
                mi += pxy * np.log(pxy / (px[i] * py[j]))
    return float(mi)

In [4]:
# ---------------------------------------------------------------------
# METRIC COMPUTATION (Raw Components only)
# ---------------------------------------------------------------------

def compute_raw_metrics(X, Hopt=0.7, sigma_H=0.15):
    """
    Compute raw Heff and M.
    NOTE: Does NOT compute final Psi because Psi requires population Z-scoring.
    """
    C, T = X.shape

    # --- 1. Hierarchical Integration (H_eff) ---
    H_vals = []
    for ch in range(C):
        try:
            h = dfa_hurst(X[ch])
        except:
            h = 0.5
        H_vals.append(h)
        
    H_raw = float(np.mean(H_vals))
    
    # Gaussian Tuning (Eq. 2 in paper)
    Heff = np.exp(-((H_raw - Hopt) ** 2) / (2.0 * sigma_H**2))

    # --- 2. Metastability (M) ---
    # Centering signal
    X_centered = X - np.mean(X, axis=1, keepdims=True)
    
    # Hilbert Transform to get Phase
    analytic_signal = hilbert(X_centered, axis=-1)
    phases = np.angle(analytic_signal)
    
    # Kuramoto Order Parameter R(t) (Eq. 3 in paper)
    R_t = np.abs(np.mean(np.exp(1j * phases), axis=0))
    
    # Metastability (Eq. 4 in paper)
    M = float(np.std(R_t))

    return {
        "H_raw": H_raw, 
        "Heff": Heff, 
        "M": M
    }

In [5]:
def summarize_experiment_results(experiment_log):
    """
    Tổng hợp kết quả từ quá trình training.
    Input: List các dictionary lưu trữ theo từng epoch.
    Output: DataFrame đã tính toán Psi chuẩn hóa (Z-score).
    """
    df = pd.DataFrame(experiment_log)
    
    # 1. Tính trung bình và độ lệch chuẩn của toàn bộ quá trình (Population stats)
    # Để dùng cho công thức Z-score (Eq. 5 trong bài báo)
    mu_H = df['Heff'].mean()
    std_H = df['Heff'].std() if df['Heff'].std() > 0 else 1e-9
    
    mu_M = df['M'].mean()
    std_M = df['M'].std() if df['M'].std() > 0 else 1e-9
    
    # 2. Tính Z-score cho từng Epoch
    df['Hz'] = (df['Heff'] - mu_H) / std_H
    df['Mz'] = (df['M'] - mu_M) / std_M
    
    # 3. Tính Psi Composite (Psi')
    # Công thức: Psi' = 0.5 * Hz + 0.5 * Mz
    df['Psi'] = 0.5 * df['Hz'] + 0.5 * df['Mz']
    
    return df

In [6]:
# ---------------------------------------------------------------------
# UTILS: EARLY STOPPING (Thêm class này vào để code chạy được)
# ---------------------------------------------------------------------
class EarlyStopping:
    def __init__(self, patience=7, delta=0, path='best_model.pt'):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'   -> EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

class FastCIFAR100(torch.utils.data.Dataset):
    def __init__(self, train=True, device='cuda'):
        # Đường dẫn gốc trỏ thẳng vào nơi chứa các file dữ liệu thô của CIFAR-100 trên Kaggle
        base_path = '/kaggle/input/datasets/fedesoriano/cifar100'
        
        if train:
            batch_file = os.path.join(base_path, 'train')
        else:
            batch_file = os.path.join(base_path, 'test')
            
        # Đọc file nhị phân bằng pickle với kiểu mã hóa latin1 chuẩn cấu trúc đại học Toronto
        with open(batch_file, 'rb') as f:
            d = pickle.load(f, encoding='latin1')
            raw_data = d['data']
            # CIFAR-100 có 2 loại nhãn: coarse_labels (20 siêu lớp) và fine_labels (100 lớp chi tiết)
            # Bài báo gốc sử dụng 100 lớp chi tiết nên ta bốc chính xác từ khóa 'fine_labels'
            all_targets = d['fine_labels']
        
        # Cấu trúc gốc của mảng d['data'] là phẳng (Flat): (N, 3072)
        # Thực hiện biến đổi hình học (Reshape) về định dạng tensor ảnh chuẩn: (N, 3, 32, 32)
        N = raw_data.shape[0]
        raw_data = raw_data.reshape(N, 3, 32, 32)
        
        # Đẩy thẳng toàn bộ mảng ma trận pixel và nhãn lên bộ nhớ VRAM của GPU
        self.data = torch.tensor(raw_data, dtype=torch.float32, device=device)
        self.targets = torch.tensor(all_targets, dtype=torch.long, device=device)
        
        # Khởi tạo các hằng số chuẩn hóa toán học đặc trưng riêng của CIFAR-100
        self.mean = torch.tensor([0.5071, 0.4867, 0.4408]).view(3, 1, 1).to(device)
        self.std = torch.tensor([0.2675, 0.2565, 0.2761]).view(3, 1, 1).to(device)
        
        self.train = train

    def __getitem__(self, index):
        # Đưa dải pixel từ [0, 255] về [0.0, 1.0]
        img = self.data[index] / 255.0
        target = self.targets[index]
        
        # Phép toán chuẩn hóa ma trận phân phối (Broadcasting tốc độ cao trên GPU)
        img = (img - self.mean) / self.std
        
        # GPU-Accelerated Augmentation (Tăng cường dữ liệu chống quá khớp tuyệt đối cho 100 lớp)
        if self.train:
            # 1. Random Horizontal Flip (Lật ảnh ngang ngẫu nhiên)
            if torch.rand(1, device=img.device) > 0.5:
                img = torch.flip(img, [2])
            
            # 2. Random Crop (Đệm phản xạ 4 pixel rồi cắt ngẫu nhiên về kích thước cũ 32x32)
            padded = torch.nn.functional.pad(img, (4, 4, 4, 4), mode='reflect')
            h_start = torch.randint(0, 9, (1,), device=img.device).item() # Chọn ngẫu nhiên tọa độ từ 0-8
            w_start = torch.randint(0, 9, (1,), device=img.device).item()
            img = padded[:, h_start:h_start+32, w_start:w_start+32]
            
        return img, target

    def __len__(self):
        return len(self.data)

In [7]:
# Hàm thiết lập Seed nghiêm ngặt trước mỗi lượt chạy
def set_deterministic_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Ép PyTorch tính toán chính xác tuyệt đối (chấp nhận giảm một chút tốc độ)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [8]:
# ============================================================================
# BLOCK 7: MULTI-SEED EXECUTION LOGIC (RESNET-101 & CIFAR-100)
# ============================================================================
if __name__ == "__main__":
    # 1. CẤU HÌNH HỆ THỐNG
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    VAL_BATCH_SIZE = 4096 
    TRAIN_BATCH_SIZE = 512  
    EPOCHS = 100          
    PATIENCE = 10         
    
    # Định nghĩa 5 hạt giống tiêu chuẩn nghiên cứu khoa học
    SEEDS = [42, 123, 1000, 1337, 2026]
    
    # Cấu trúc lưu trữ đóng gói dữ liệu tổng hợp liên-seed
    all_experiments_summary = {}
    aggregated_metrics = {
        "best_accuracy": [], 
        "mean_Heff": [], 
        "std_Heff": [],
        "sigma_Psi": [], 
        "r_Hz_Mz": [], 
        "r_psi_acc": []
    }

    print(f"🚀 Khởi động chiến dịch huấn luyện ResNet-101 trên {len(SEEDS)} hạt giống...")

    # VÒNG LẶP ĐA HẠT GIỐNG (MULTI-SEED LOOP)
    for run_idx, current_seed in enumerate(SEEDS):
        print("\n" + "="*70)
        print(f"🔥 LƯỢT CHẠY {run_idx + 1}/{len(SEEDS)} | SEED KHỞI TẠO: {current_seed}")
        print("="*70)
        
        # ------------------------------------------------------------------------
        # CHỐT CHẶN BẮT BUỘC TẠI ĐẦU MỖI SEED: DỌN SẠCH VÀ RESET TOÀN DIỆN
        # ------------------------------------------------------------------------
        if 'activations' in locals() or 'activations' in globals():
            if isinstance(activations, dict):
                activations.clear()  # Xóa sạch các phần tử kích hoạt tensor ẩn của seed trước
            del activations          # Hủy hoàn toàn con trỏ tham chiếu biến để tránh leak dữ liệu
        
        gc.collect()
        if torch.cuda.is_available(): 
            torch.cuda.empty_cache()
            
        # Cố định cấu trúc ngẫu nhiên nghiêm ngặt cho lượt chạy hiện tại
        set_deterministic_seeds(current_seed)

        # 2. DATA LOAD (Đọc trực tiếp từ đĩa cứng Kaggle lên VRAM qua FastCIFAR100)
        print("Loading CIFAR-100 directly to H100 VRAM...")
        trainset_fast = FastCIFAR100(train=True, device=DEVICE)
        valset_fast = FastCIFAR100(train=False, device=DEVICE)

        trainloader = torch.utils.data.DataLoader(trainset_fast, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
        valloader = torch.utils.data.DataLoader(valset_fast, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0)

        # 3. INITIALIZE MODEL & OPTIMIZER FOR CURRENT SEED (num_classes=100)
        print("Initializing ResNet-101...")
        model = torchvision.models.resnet101(num_classes=100).to(DEVICE)
        
        LR = 0.1
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5) 
        
        # Đặt tên checkpoint độc lập theo Seed để tránh xung đột ghi đè
        early_stopping = EarlyStopping(patience=PATIENCE, path=f'best_resnet101_seed_{current_seed}.pt')
        
        # Gắn hệ thống Hook đa tầng của ResNet-101
        activations = setup_resnet_hooks(model)

        # 4. INTRA-SEED TRAINING LOOP
        history = []
        for epoch in range(EPOCHS):
            # --- TRAIN LAYER ---
            model.train()
            train_loss, correct, total = 0, 0, 0
            for inputs, targets in trainloader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
            
            avg_train_loss = train_loss / len(trainloader)
            train_acc = 100. * correct / total

            # --- VALIDATE LAYER ---
            model.eval()
            val_loss, val_correct, val_total = 0, 0, 0
            with torch.no_grad():
                for inputs, targets in valloader:
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item()
                    _, predicted = outputs.max(1)
                    val_total += targets.size(0)
                    val_correct += predicted.eq(targets).sum().item()
            
            avg_val_loss = val_loss / len(valloader)
            val_acc = 100. * val_correct / val_total
            scheduler.step(avg_val_loss)

            # --- TÍNH PSI ĐA TẦNG (ALGORITHM 1) ---
            data_iter = iter(valloader)
            inputs_psi, targets_psi = next(data_iter)
            with torch.no_grad():
                _ = model(inputs_psi) 
            
            multiscale_matrix = gather_multiscale_activations(activations, samples_per_layer=32)
            cnn_signals = prepare_cnn_activations(multiscale_matrix, targets_psi, smooth_window=15)
            metrics = compute_raw_metrics(cnn_signals)

            log = {
                "epoch": epoch + 1, "train_loss": avg_train_loss, "val_loss": avg_val_loss,
                "val_acc": val_acc, "Heff": metrics["Heff"], "M": metrics["M"], "H_raw": metrics["H_raw"]
            }
            history.append(log)

            if (epoch + 1) % 10 == 0:
                print(f"   Ep {epoch+1:03d} | Loss:{avg_val_loss:.3f} | Acc:{val_acc:.2f}% | Heff:{metrics['Heff']:.2f} | M:{metrics['M']:.3f}")

            early_stopping(avg_val_loss, model)
            if early_stopping.early_stop:
                print(f"   🛑 Early stopping kích hoạt tại epoch {epoch+1}")
                break

        # 5. PHÂN TÍCH KẾT QUẢ RIÊNG CỦA HẠT GIỐNG HIỆN TẠI (TÍNH THÔ)
        df_results = summarize_experiment_results(history)
        df_valid = df_results.dropna(subset=['Heff', 'M', 'Psi', 'Hz', 'Mz']).copy()
        
        if len(df_valid) > 1:
            m_heff, s_heff = df_valid['Heff'].mean(), df_valid['Heff'].std()
            sig_psi = df_valid['Psi'].std()
            b_acc = df_results['val_acc'].max()
            
            # Tính tương quan thô, chấp nhận sinh ra NaN nếu chuỗi đứng im
            r_hz_mz = df_valid['Hz'].corr(df_valid['Mz'])
            r_psi_acc = df_valid['Psi'].corr(df_valid['val_acc'])
            
            # Thu thập trực tiếp vào mảng gom liên-seed
            aggregated_metrics["best_accuracy"].append(b_acc)
            aggregated_metrics["mean_Heff"].append(m_heff)
            aggregated_metrics["std_Heff"].append(s_heff)
            aggregated_metrics["sigma_Psi"].append(sig_psi)
            aggregated_metrics["r_Hz_Mz"].append(r_hz_mz)
            aggregated_metrics["r_psi_acc"].append(r_psi_acc)
            
            # Đóng gói chuỗi string báo cáo thô cho seed hiện tại
            seed_string = f"{b_acc:.1f}% | {m_heff:.3f} ± {s_heff:.3f} | {sig_psi:.4f} | {r_hz_mz:.3f} | {r_psi_acc:.3f}"
            all_experiments_summary[f"seed_{current_seed}"] = {
                "best_accuracy": float(b_acc), "mean_Heff": float(m_heff), "std_Heff": float(s_heff),
                "sigma_Psi": float(sig_psi), "r_Hz_Mz": float(r_hz_mz) if not np.isnan(r_hz_mz) else "NaN", 
                "r_Psi_acc": float(r_psi_acc) if not np.isnan(r_psi_acc) else "NaN",
                "paper_ready_string": seed_string
            }
            
            # Xuất bản ghi thời gian epoch sạch sẽ ra file CSV riêng biệt
            df_results.to_csv(f"results_resnet101_cifar100_seed_{current_seed}.csv", index=False)
            print(f"✅ Đã lưu lịch sử epoch của Seed {current_seed} vào file CSV.")
        
        # Hủy các đối tượng mô hình thô ở cuối lượt chạy để chuẩn bị dọn dẹp triệt để tại đầu vòng lặp kế tiếp
        del model, optimizer, trainloader, valloader

    # ============================================================================
    # 6. TÍNH TOÁN TOÀN CỤC LIÊN-SEED VÀ XUẤT FILE CẤU HÌNH TÓM TẮT
    # ============================================================================
    print("\n" + "🏁" * 15 + " KẾT LUẬN TOÀN DIỆN DIỄN ĐÀN 5 SEEDS (RESNET-101) " + "🏁" * 15)
    
    final_report = {
        "metadata": {"architecture": "ResNet-101", "dataset": "CIFAR-100", "seeds_tested": SEEDS}
    }
    
    summary_outputs = {}
    for k, values in aggregated_metrics.items():
        # Lọc bỏ các phần tử NaN (nếu có) khi tính toán thống kê mô tả cuối cùng
        valid_vals = [v for v in values if not np.isnan(v)]
        if len(valid_vals) > 0:
            arr = np.array(valid_vals)
            summary_outputs[k] = {"mean": float(arr.mean()), "std": float(arr.std())}
        else:
            summary_outputs[k] = {"mean": "NaN", "std": "NaN"}
            
    final_report["global_statistics"] = summary_outputs
    final_report["individual_runs"] = all_experiments_summary

    # Chốt chặn định dạng an toàn: Kiểm tra kiểu dữ liệu để tránh sập chuỗi nếu dính rác NaN
    fmt_r_hz_mean = f"{summary_outputs['r_Hz_Mz']['mean']:.3f}" if isinstance(summary_outputs['r_Hz_Mz']['mean'], float) else str(summary_outputs['r_Hz_Mz']['mean'])
    fmt_r_hz_std  = f"{summary_outputs['r_Hz_Mz']['std']:.3f}" if isinstance(summary_outputs['r_Hz_Mz']['std'], float) else str(summary_outputs['r_Hz_Mz']['std'])
    fmt_r_psi_mean = f"{summary_outputs['r_psi_acc']['mean']:.3f}" if isinstance(summary_outputs['r_psi_acc']['mean'], float) else str(summary_outputs['r_psi_acc']['mean'])
    fmt_r_psi_std  = f"{summary_outputs['r_psi_acc']['std']:.3f}" if isinstance(summary_outputs['r_psi_acc']['std'], float) else str(summary_outputs['r_psi_acc']['std'])

    # Chuỗi văn bản phục vụ chèn trực tiếp vào Latex hoặc bảng biểu báo cáo bài báo
    super_paper_line = (
        f"ResNet-101 CIFAR-100 | "
        f"Best Acc: {summary_outputs['best_accuracy']['mean']:.1f}±{summary_outputs['best_accuracy']['std']:.1f}% | "
        f"Heff: {summary_outputs['mean_Heff']['mean']:.3f}±{summary_outputs['mean_Heff']['std']:.3f} | "
        f"σ_Ψ: {summary_outputs['sigma_Psi']['mean']:.4f}±{summary_outputs['sigma_Psi']['std']:.4f} | "
        f"r(Hz,Mz): {fmt_r_hz_mean}±{fmt_r_hz_std} | "
        f"r(Ψ,acc): {fmt_r_psi_mean}±{fmt_r_psi_std}"
    )
    final_report["super_paper_ready_string"] = super_paper_line

    print("\n📊 DÒNG TỔNG HỢP SAU LƯỢT CHẠY MULTI-SEED:")
    print("-" * 100)
    print(super_paper_line)
    print("-" * 100)

    # Xuất file JSON cấu hình độc lập duy nhất lưu giữ toàn bộ metadata của 5 hạt giống
    with open("multi_seed_summary_resnet101_cifar100.json", "w", encoding="utf-8") as f:
        json.dump(final_report, f, ensure_ascii=False, indent=4)
    print("\n💾 Đã lưu file cấu hình tóm tắt tối cao: multi_seed_summary_resnet101_cifar100.json")

🚀 Khởi động chiến dịch huấn luyện ResNet-101 trên 5 hạt giống...

🔥 LƯỢT CHẠY 1/5 | SEED KHỞI TẠO: 42
Loading CIFAR-100 directly to H100 VRAM...
Initializing ResNet-101...
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 1 out of 10
   Ep 010 | Loss:3.265 | Acc:19.95% | Heff:0.53 | M:0.103
   -> EarlyStopping counter: 1 out of 10
   Ep 020 | Loss:2.627 | Acc:33.43% | Heff:0.41 | M:0.076
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 2 out of 10
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 1 out of 10
   Ep 030 | Loss:2.381 | Acc:38.56% | Heff:0.41 | M:0.072
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 2 out of 10
   -> EarlyStopping counter: 3 out of 10
   -> EarlyStopping counter: 4 out of 10
   Ep 040 | Loss:2.187 | Acc:42.61% | Heff:0.43 | M:0.066
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 2 out of 10
   -> EarlyStopping co